# 06 Gemma 4 E4B Context Pruning Experiment (Colab)

This notebook runs the context-pruning generation experiment with the recommended Gemma model.

Default run:
- model: `google/gemma-4-E4B-it`
- variant: `B_pruned_context_by_question_type`
- eval set: random 50 questions sampled from the canonical full eval CSVs

The notebook builds `data/eval/gemma_full_eval_sample_50.csv` from `data/eval/eval_batch_*.csv` and runs generation only for those 50 questions.


In [20]:
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')

PREDICTION_REL = Path('outputs/predictions/best_variant_predictions.jsonl')
FULL_EVAL_DIR_REL = Path('data/eval')
FULL_EVAL_GLOB = 'eval_batch_*.csv'
SAMPLE_EVAL_REL = Path('data/eval/gemma_full_eval_sample_50.csv')
SAMPLE_SIZE = 50
SAMPLE_RANDOM_SEED = 42
EVAL_REL = SAMPLE_EVAL_REL
CHUNK_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'google/gemma-4-E4B-it'
FALLBACK_MODEL_NAME = 'google/gemma-4-E2B-it'
MAX_NEW_TOKENS = 384
RUN_LIMIT = 50
RUN_VARIANTS = ['B_pruned_context_by_question_type']

RUN_CONTEXT_ONLY_DRY_RUN = True
CONTEXT_ONLY_DRY_RUN_LIMIT = 2
RUN_GENERATION = True

LOCAL_OUTPUT_ROOT = PROJECT_DIR / 'outputs/gemma_context_experiments'
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'gemma_context_experiments'
DRY_RUN_NAME = 'gemma4_e4b_full50_dry_run'
GENERATION_RUN_NAME = 'gemma4_e4b_b_pruned_full50'

print('branch:', BRANCH)
print('model:', MODEL_NAME)
print('fallback:', FALLBACK_MODEL_NAME)
print('full_eval_dir:', FULL_EVAL_DIR_REL)
print('sample_eval:', EVAL_REL)
print('sample_size:', SAMPLE_SIZE)
print('sample_seed:', SAMPLE_RANDOM_SEED)
print('predictions:', PREDICTION_REL)
print('chunks:', CHUNK_REL)
print('source_store:', SOURCE_STORE_REL)
print('run_limit:', RUN_LIMIT)
print('variants:', RUN_VARIANTS)


branch: colab-generation
model: google/gemma-4-E4B-it
fallback: google/gemma-4-E2B-it
full_eval_dir: data/eval
sample_eval: data/eval/gemma_full_eval_sample_50.csv
sample_size: 50
sample_seed: 42
predictions: outputs/predictions/best_variant_predictions.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
source_store: data/source_store_v2_690.jsonl
run_limit: 50
variants: ['B_pruned_context_by_question_type']


## 1. Check GPU

In [21]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Colab GPU is not enabled. Select Runtime > Change runtime type > GPU.')

!nvidia-smi

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA L4
Thu May 28 06:12:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   52C    P8             16W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                

## 2. Clone or pull colab-generation

In [22]:
import os
import subprocess

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)

cwd: /content/chatbot


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## 3. Install dependencies

Gemma 4 may need a recent Transformers version, so this cell updates the HF stack.

In [23]:
# Lightweight install for Gemma generation only.
# Do not install the full requirements.txt here: it can downgrade numpy in Colab.
%pip uninstall -y -q torchvision
%pip install -q -U "transformers>=4.57.0" accelerate sentencepiece huggingface_hub "protobuf<7"

import os
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
print('HF stack installed. If you already ran the old install cell in this runtime, restart the runtime once and rerun from the first cell.')


HF stack installed. If you already ran the old install cell in this runtime, restart the runtime once and rerun from the first cell.


## 4. Hugging Face login and model access check

If the model is gated, accept the model license on Hugging Face and add `HF_TOKEN` to Colab Secrets.

In [24]:
import os

os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
hf_token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or hf_token
except Exception:
    pass

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('HF_TOKEN login complete')
else:
    print('HF_TOKEN not found. If model access fails, add HF_TOKEN to Colab Secrets or run huggingface_hub.login().')

from huggingface_hub import model_info

try:
    info = model_info(MODEL_NAME, token=hf_token)
    print('model access ok:', MODEL_NAME)
    print('model id:', info.modelId)
    print('private:', info.private)
    print('gated:', getattr(info, 'gated', None))
except Exception as exc:
    print('model access failed:', MODEL_NAME)
    print(type(exc).__name__, exc)
    print('Fallback option:', FALLBACK_MODEL_NAME)
    raise


HF_TOKEN not found. If model access fails, add HF_TOKEN to Colab Secrets or run huggingface_hub.login().
model access ok: google/gemma-4-E4B-it
model id: google/gemma-4-E4B-it
private: False
gated: False


## 5. Mount Google Drive

In [25]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive input root:', DRIVE_INPUT_ROOT)
print('Drive output root:', DRIVE_OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive input root: /content/drive/MyDrive/chatbot_colab_inputs
Drive output root: /content/drive/MyDrive/chatbot_colab_outputs


## 6. Copy inputs and build a 50-question eval sample

This cell copies the full canonical eval CSVs, samples 50 questions with a fixed seed, and writes `data/eval/gemma_full_eval_sample_50.csv`.


In [26]:
import csv
import random
import shutil

def resolve_drive_prediction_path(path):
    if path.exists():
        return path
    if not path.suffix:
        jsonl_path = path.with_suffix('.jsonl')
        if jsonl_path.exists():
            return jsonl_path
    return path

DRIVE_PREDICTION_PATH = resolve_drive_prediction_path(DRIVE_INPUT_ROOT / PREDICTION_REL)
LOCAL_PREDICTION_PATH = PROJECT_DIR / PREDICTION_REL
if DRIVE_PREDICTION_PATH.suffix and LOCAL_PREDICTION_PATH.suffix != DRIVE_PREDICTION_PATH.suffix:
    LOCAL_PREDICTION_PATH = LOCAL_PREDICTION_PATH.with_suffix(DRIVE_PREDICTION_PATH.suffix)
    PREDICTION_REL = LOCAL_PREDICTION_PATH.relative_to(PROJECT_DIR)

required_inputs = [
    ('eval_dir', DRIVE_INPUT_ROOT / FULL_EVAL_DIR_REL),
    ('predictions', DRIVE_PREDICTION_PATH),
    ('chunks', DRIVE_INPUT_ROOT / CHUNK_REL),
    ('source_store', DRIVE_INPUT_ROOT / SOURCE_STORE_REL),
]
missing = [(name, path) for name, path in required_inputs if not path.exists()]
if missing:
    detail = '\n'.join(f'- {name}: {path}' for name, path in missing)
    raise FileNotFoundError('Missing Drive input path(s):\n' + detail)

local_eval_dir = PROJECT_DIR / FULL_EVAL_DIR_REL
local_eval_dir.mkdir(parents=True, exist_ok=True)
drive_eval_paths = sorted((DRIVE_INPUT_ROOT / FULL_EVAL_DIR_REL).glob(FULL_EVAL_GLOB))
if not drive_eval_paths:
    raise FileNotFoundError(f'No eval CSVs found: {DRIVE_INPUT_ROOT / FULL_EVAL_DIR_REL / FULL_EVAL_GLOB}')

for src in drive_eval_paths:
    dst = local_eval_dir / src.name
    shutil.copy2(src, dst)
print('copied eval csv count:', len(drive_eval_paths))

copy_pairs = [
    (DRIVE_PREDICTION_PATH, LOCAL_PREDICTION_PATH),
    (DRIVE_INPUT_ROOT / CHUNK_REL, PROJECT_DIR / CHUNK_REL),
    (DRIVE_INPUT_ROOT / SOURCE_STORE_REL, PROJECT_DIR / SOURCE_STORE_REL),
]
for src, dst in copy_pairs:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied: {src} -> {dst} ({dst.stat().st_size:,} bytes)')

rows = []
fieldnames = []
for csv_path in sorted(local_eval_dir.glob(FULL_EVAL_GLOB)):
    with csv_path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        if reader.fieldnames:
            for name in reader.fieldnames:
                if name not in fieldnames:
                    fieldnames.append(name)
        for row in reader:
            row['source_eval_file'] = csv_path.name
            rows.append(row)
if 'source_eval_file' not in fieldnames:
    fieldnames.append('source_eval_file')
if len(rows) < SAMPLE_SIZE:
    raise ValueError(f'Not enough eval rows to sample {SAMPLE_SIZE}: {len(rows)}')

rng = random.Random(SAMPLE_RANDOM_SEED)
sampled_rows = rows[:]
rng.shuffle(sampled_rows)
sampled_rows = sampled_rows[:SAMPLE_SIZE]

sample_path = PROJECT_DIR / SAMPLE_EVAL_REL
sample_path.parent.mkdir(parents=True, exist_ok=True)
with sample_path.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sampled_rows)

print('full eval rows:', len(rows))
print('sample rows:', len(sampled_rows))
print('sample eval:', sample_path)
print('sample ids:', [row.get('id') or row.get('question_id') for row in sampled_rows[:10]])
print('resolved prediction:', PREDICTION_REL)

script_path = PROJECT_DIR / 'experiments/context_pruning_experiment.py'
if not script_path.exists():
    raise FileNotFoundError(f'Experiment script is missing. Push/pull colab-generation first: {script_path}')
print('script:', script_path)


copied eval csv count: 38
copied: /content/drive/MyDrive/chatbot_colab_inputs/outputs/predictions/best_variant_predictions.jsonl -> /content/chatbot/outputs/predictions/best_variant_predictions.jsonl (8,222,469 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl -> /content/chatbot/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl (368,289,817 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/data/source_store_v2_690.jsonl -> /content/chatbot/data/source_store_v2_690.jsonl (553,876,319 bytes)
full eval rows: 1100
sample rows: 50
sample eval: /content/chatbot/data/eval/gemma_full_eval_sample_50.csv
sample ids: ['Q047', 'Q075', 'Q288', 'Q037', 'Q049', 'Q003', 'Q063', 'Q037', 'Q040', 'Q024']
resolved prediction: outputs/predictions/best_variant_predictions.jsonl
script: /content/chatbot/experiments/context_pruning_experiment.py


## 7. Runner helper

In [27]:
import sys

CREATED_OUTPUT_DIRS = []

def list_experiment_outputs():
    LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return {path.resolve() for path in LOCAL_OUTPUT_ROOT.iterdir() if path.is_dir()}

def run_gemma_experiment(*, context_only: bool, limit: int, run_name: str):
    before = list_experiment_outputs()
    cmd = [
        sys.executable,
        str(PROJECT_DIR / 'experiments/context_pruning_experiment.py'),
        '--predictions', str(PREDICTION_REL),
        '--eval-csv', str(EVAL_REL),
        '--chunks', str(CHUNK_REL),
        '--source-store', str(SOURCE_STORE_REL),
        '--output-root', str(LOCAL_OUTPUT_ROOT.relative_to(PROJECT_DIR)),
        '--run-name', run_name,
        '--model-name', MODEL_NAME,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--limit', str(limit),
    ]
    if context_only:
        cmd.append('--context-only')
    for variant in RUN_VARIANTS:
        cmd.extend(['--variant', variant])
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    after = list_experiment_outputs()
    created = sorted(after - before, key=lambda path: path.stat().st_mtime)
    if not created:
        raise RuntimeError('Could not find the new output directory.')
    CREATED_OUTPUT_DIRS.extend(created)
    print('created output:', created[-1])
    return created[-1]

## 8. Context-only dry run

This validates context construction without loading the model.

In [28]:
if RUN_CONTEXT_ONLY_DRY_RUN:
    dry_output_dir = run_gemma_experiment(
        context_only=True,
        limit=CONTEXT_ONLY_DRY_RUN_LIMIT,
        run_name=DRY_RUN_NAME,
    )
else:
    dry_output_dir = None
    print('Context-only dry run skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/gemma_full_eval_sample_50.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/gemma_context_experiments --run-name gemma4_e4b_full50_dry_run --model-name google/gemma-4-E4B-it --max-new-tokens 384 --limit 2 --context-only --variant B_pruned_context_by_question_type
created output: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_full50_dry_run_20260528_061302


## 9. Run Gemma generation

This run uses 30 questions from the representative 50-question sample eval CSV.

In [29]:
if RUN_GENERATION:
    generation_output_dir = run_gemma_experiment(
        context_only=False,
        limit=RUN_LIMIT,
        run_name=GENERATION_RUN_NAME,
    )
else:
    generation_output_dir = None
    print('Generation skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/gemma_full_eval_sample_50.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/gemma_context_experiments --run-name gemma4_e4b_b_pruned_full50 --model-name google/gemma-4-E4B-it --max-new-tokens 384 --limit 50 --variant B_pruned_context_by_question_type
created output: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307


## 10. Inspect results

In [30]:
import csv
import html as html_lib
from IPython.display import HTML, display

latest_output_dir = generation_output_dir or dry_output_dir
if latest_output_dir is None:
    raise RuntimeError('No output directory to inspect.')

metrics_path = latest_output_dir / 'context_pruning_metrics.csv'
review_path = latest_output_dir / 'context_pruning_review.csv'
summary_path = latest_output_dir / 'context_pruning_summary.md'
results_path = latest_output_dir / 'context_pruning_results.jsonl'

print('results:', results_path)
print('review:', review_path)
print('metrics:', metrics_path)
print('summary:', summary_path)

def read_csv_preview(path, limit=None):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        rows = []
        for idx, row in enumerate(reader):
            if limit is not None and idx >= limit:
                break
            rows.append(row)
    return rows

def display_rows(rows, title, max_cols=12):
    print(f'{title}: {len(rows)} row(s) shown')
    if not rows:
        return
    columns = list(rows[0].keys())[:max_cols]
    header = ''.join(f'<th>{html_lib.escape(col)}</th>' for col in columns)
    body = []
    for row in rows:
        cells = ''.join(
            '<td style="max-width:260px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
            + html_lib.escape(str(row.get(col, ''))) + '</td>'
            for col in columns
        )
        body.append(f'<tr>{cells}</tr>')
    display(HTML(
        '<div style="overflow:auto;max-height:420px">'
        f'<table border="1" style="border-collapse:collapse;font-size:12px">'
        f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'
        '</div>'
    ))

metrics_rows = read_csv_preview(metrics_path)
review_rows = read_csv_preview(review_path, limit=10)
display_rows(metrics_rows, 'metrics')
display_rows(review_rows, 'review preview')

results: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307/context_pruning_results.jsonl
review: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307/context_pruning_review.csv
metrics: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307/context_pruning_metrics.csv
summary: /content/chatbot/outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307/context_pruning_summary.md
metrics: 1 row(s) shown


answer_available_rate,answerable_rate,avg_context_chars,avg_direct_evidence_count,avg_noisy_evidence_count,avg_reference_evidence_count,avg_source_store_items,avg_supporting_evidence_count,citation_valid_rate,failure_tag_counts,gold_signal_present_rate,median_context_chars
1.0,0.07142857142857142,5418.5,3.380952380952381,1.0476190476190477,0.5714285714285714,0.0,1.9047619047619047,1.0,"{""target_doc_coverage_missing"": 19, ""gt_numeric_mismatch"": 1, ""gt_expected_answer_but_model_not_found"": 16, ""unit_conversion_corrected"": 1, ""source_numeric_missing"": 6, ""citation_wrong_target"": 3, ""wrong_target_field_selection"": 1, ""llm_invalid_json"": 2}",0.42857142857142855,5690.5


review preview: 10 row(s) shown


context_char_count,failure_tags,failure_type,generated_answer,gold_answer,gold_signal_present,manual_correct,question,question_id,question_type,retrieved_docs_top5,review_note
5195,"[""target_doc_coverage_missing""]",,제공된 Context 내에서는 두 사업 중 어느 것이 해외 지역을 대상으로 기술 인프라를 이전하거나 구축하는 사업인지에 대한 정보가 명확하게 나와 있지 않습니다.,주 대상이 해외 지역인 본 사업은 코이카 전기조달에서 발주한 '우즈베키스탄 열린 의정활동 상하원 국회 방송시스템 구축' 용역 사업입니다.,False,,"한국가스공사의 '차세대 ERP 구축'과 코이카의 '우즈베키스탄 방송시스템 사업' 중, 해외 지역을 대상으로 기술 인프라를 이전하거나 구축하는 사업은 무엇입니까?",Q047,business_type,"[""KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp"", ""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"", ""KOICA 전자조달_[지문] [국제] (재공고)우즈베키스탄 ICT기반의 수자원정보.hwp"", ""한국농어촌공사_차세대 ERP 통합업무포털시스템 구축 용역.hwp"", ""한국기계연구원_(재공고)기계(연) 차세대 통합정보시스템 구축.hwp""]",
7537,[],,제공된 Context 내에서는 해당 사안에 대한 답변을 찾을 수 없습니다.,"질문에 제시된 벌점 관련 정보는 제시되지 않았습니다. 보안위규 위반 처리관련 언급은 있어도, 벌점 관련 정보는 없습니다.",True,,"중앙선거관리위원회 2025년도 행정정보시스템 위탁운영사업을 수행하는 상주 인력이 내부 지시 불이행 등으로 벌점 8점이 부과된 상태입니다. 이후 상주구역 내에서 무단으로 사진을 촬영하다가 보안 위규에 적발되었다면, 해당 인력은 PM 판단과 관계없이 누적 조치로 인한 즉각적인 퇴출 및 인력 교체 대상이 됩니까?",Q075,summary,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf"", ""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf"", ""고려대학교_Student Success Center 시스템 개선 및 교과-비교과 통합관리시스.pdf"", ""고려대학교_[재공지] 고려대학교 공간관리 통합시스템 구축 사업.hwp"", ""고려대학교_고려대학교 공간관리 통합시스템 구축 사업.hwp""]",
4092,"[""gt_numeric_mismatch""]",,"두 금액을 합산하면 총 598,000,000원입니다. 계산 기준은 282,000,000원와 316,000,000원입니다.","동일 부처 내 병행되는 두 국방 데이터 사업의 합산 총액은 599,100,000원입니다.",False,,국방과학연구소의 기록관리 환경 사업비용(약 2.82억)과 자료전송시스템 고도화 사업비용(약 3.16억)을 합친 총액은 얼마입니까?,Q288,budget,"[""국방과학연구소_대용량 자료전송시스템 고도화.hwp"", ""국방과학연구소_기록관리시스템 통합 활용 및 보안 환경 구축.hwp"", ""국방과학연구소_기록관리시스템 통합 관리 환경 구축.hwp"", ""국방과학연구소_e-감사시스템 웹 호환성 및 기능보완.hwp"", ""오송첨단의료산업진흥재단_2024년 통합정보시스템 고도화 사업.hwp""]",
2052,[],,제공된 Context 내에서는 경기도 안양시 호계체육관 예약시스템 사업의 원격지 접속 시 지정 단말기 로그기록 보관 기간이나 경기도사회서비스원 연간 유지보수 사업에서 비공개 처리된 하드웨어 구성도를 열람하기 위해 입찰참가업체가 발주기관 방문 시 제출해야 하는 서류에 대한 정보가 없습니다.,안양시 사업에서 원격지 개발 지정 단말기의 로그기록은 '1년 이상' 보관해야 합니다. 경기도사회서비스원 사업에서는 하드웨어 구성도 열람을 위해 '보안각서'를 제출해야 합니다.,True,,"경기도 안양시 호계체육관 예약시스템 사업과 경기도사회서비스원 연간 유지보수 사업은 규정과 관련된 접근 통제를 명시하고 있습니다. 안양시 사업에서 원격지 접속 시 지정 단말기 로그기록은 얼마 동안 보관해야 하며, 경기도사회서비스원 사업에서 비공개 처리된 하드웨어 구성도를 열람하기 위해 입찰참가업체가 발주기관 방문 시 제출해야 하는 서류는 무엇입니까?",Q037,summary,"[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp""]",
5218,"[""gt_expected_answer_but_model_not_found""]",,제공된 컨텍스트에는 법적인 문제를 사전에 명확히 방지하기 위해 최종 산출물 제출 시 반드시 포함해야 하는 특정 관련 증빙 서류 목록이 명시되어 있지 않습니다.,"제안사는 시스템 웹사이트 등에 사용된 저작권이 걸려있는 신문기사나 이미지 등에 대한 합법적 사용을 증명하기 위해, 1차 저작권자로부터 발급받은 '저작권 허락동의서'를 문서 형태로 확보하여 반드시 제출해야 합니다.",False,,그렇다면 이러한 법적인 문제를 사전에 명확히 방지하기 위해 제안사가 최종 산출물 제출 시 반드시 포함하여 제출해야 하는 관련 증빙 서류는 무엇입니까?,Q049,summary,"[""나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp"", ""인천광역시_도시계획위원회 통합관리시스템 구축용역.hwp"", ""인천광역시_IDC 통합운영관리시스템 고도화 및 클라우드 포털 구축 사업.hwp"", ""화성시청_용역 입찰공고[화성시 도시계획위원회 통합관리시스템 구축 .hwp"", ""서울특별시서울의료원_2025년 스마트 호스피털 서비스 시스템 유지운영 .hwp""]",
7595,"[""target_doc_coverage_missing"", ""gt_expected_answer_but_model_not_found""]",,"제공된 Context 내에서는 요청하신 세 사업(""(사)벤처기업협회 벤처확인종합관리시스템 고도화"", ""(사)부산국제영화제 온라인서비스 재개발"", ""(사)한국대학스포츠협의회 경기기록 관리시스템 개발"")의 예산 정보를 찾을 수 없습니다.","예산 규모가 가장 큰 사업은 352,000,000원인 (사)벤처기업협회 사업이며, 다음은 243,000,000원인 (사)부산국제영화제 사업, 가장 작은 규모는 150,000,000원인 (사)한국대학스포츠협의회 사업입니다.",False,,"2024년에 발주된 ""(사)벤처기업협회 벤처확인종합관리시스템 고도화"", ""(사)부산국제영화제 온라인서비스 재개발"", ""(사)한국대학스포츠협의회 경기기록 관리시스템 개발"" 세 사업의 예산을 비교할 때, 예산 규모가 가장 큰 사업부터 순서대로 나열하고 각각의 사업예산을 작성해 주세요.",Q003,summary,"[""KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp"", ""KOICA 전자조달_[지문] [국제] (재공고)우즈베키스탄 ICT기반의 수자원정보.hwp"", ""KOICA 전자조달_[지문] [국제] 르완다 통합적 일자리 정보체계 구축사업 시.hwp"", ""KOICA 전자조달_[지문] [국제] 라오스 항만관리 정보화 시스템(Port-MIS) 구축.hwp"", ""KOICA 전자조달_[지문] 인도적지원 정보시스템 기본설계(ISMP) 용역.hwp""]",
2115,[],,제공된 Context 내에서는 3초 이내 성능 응답 기준이 면제되는 예외 상황에 대한 파일 크기 및 동시 접속자 수 측면의 구체적인 예외 허용 기준을 확인할 수 없습니다.,"제안요청서의 사용자 화면 디스플레이 성능 예외 조항에 따르면 첫째, 파일 크기 측면에서 1개 이상의 큰 이미지(500KB 이상) 또는 동영상을 포함하고 있는 디스플레이 페이지일 때 예외가 허용됩니다. 둘째, 동시 접속자 수 측면에서 시스템을 사용하는 사용자가 동시 사용자 용량의 90%를 초과하는 과부하 상태일 때 이 3초 이내 디스플레이 기준이 적용되지 않도록 구체적으로 설계되어 있습니다.",Tr

## 11. Copy outputs to Drive

In [31]:
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

copied_dirs = []
for local_dir in CREATED_OUTPUT_DIRS:
    dst = DRIVE_EXPERIMENT_ROOT / local_dir.name
    if dst.exists():
        raise FileExistsError(f'Drive output directory already exists. Not overwriting: {dst}')
    shutil.copytree(local_dir, dst)
    copied_dirs.append(dst)
    print('copied output to Drive:', dst)

if not copied_dirs:
    print('No new output directory to copy.')
else:
    print('Drive output dirs:')
    for path in copied_dirs:
        print('-', path)

copied output to Drive: /content/drive/MyDrive/chatbot_colab_outputs/gemma_context_experiments/gemma4_e4b_full50_dry_run_20260528_061302
copied output to Drive: /content/drive/MyDrive/chatbot_colab_outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307
Drive output dirs:
- /content/drive/MyDrive/chatbot_colab_outputs/gemma_context_experiments/gemma4_e4b_full50_dry_run_20260528_061302
- /content/drive/MyDrive/chatbot_colab_outputs/gemma_context_experiments/gemma4_e4b_b_pruned_full50_20260528_061307


## Review files

- `context_pruning_review.csv`: manual review file with `manual_correct`, `failure_type`, `review_note`
- `context_pruning_metrics.csv`: automatic proxy metrics for the Gemma run
- `context_pruning_summary.md`: summary and failure examples
- `context_pruning_results.jsonl`: detailed generation records with `used_context`

`RUN_LIMIT = 50` runs the sampled 50-question eval CSV.